# ⚙️ W6: OEE & 생산성 트렌드 분석
**hexa-1 — Week 6** | ⏱️ 60분 | 🎯 OEE 보고서 + 트렌드 차트

OEE = 가동률(Availability) × 성능률(Performance) × 양품률(Quality)

## Step 0: 환경 설정

In [ ]:
import subprocess
subprocess.run(['apt-get', '-qq', '-y', 'install', 'fonts-nanum'], capture_output=True)
!pip install -q pandas matplotlib
import pandas as pd, matplotlib.pyplot as plt, matplotlib.font_manager as fm
fm._load_fontmanager(try_read_cache=False)
nanum = next((f.fname for f in fm.fontManager.ttflist if 'Nanum' in f.name), None)
if nanum: plt.rcParams['font.family'] = fm.FontProperties(fname=nanum).get_name()
plt.rcParams['axes.unicode_minus'] = False
print('✅ 환경 설정 완료')


## Step 1: 공장 정보 입력 (✏️ 수정)

In [ ]:
FACTORY = {'name': '✏️ [교육 기업명]', 'line': '✏️ [라인명]', 'oee_target': 85.0}
print('✅ 목표 OEE:', FACTORY['oee_target'], '%')


## Step 2: 샘플 데이터 생성 (또는 CSV 업로드)

In [ ]:
import numpy as np, io
np.random.seed(42)
n = 30
df = pd.DataFrame({
    '날짜': pd.date_range('2026-01-01', periods=n, freq='D'),
    '계획시간': [480]*n,
    '가동시간': np.random.randint(400, 480, n),
    '총생산량': np.random.randint(800, 1000, n),
    '불량수': np.random.randint(5, 40, n),
    '이상시간': np.random.randint(5, 30, n),
})

# 실제 CSV 업로드 옵션
try:
    from google.colab import files
    print('💡 내 데이터 업로드 가능 (컬럼: 날짜, 계획시간, 가동시간, 총생산량, 불량수, 이상시간)')
    uploaded = files.upload()
    if uploaded:
        df = pd.read_csv(io.BytesIO(list(uploaded.values())[0]), encoding='utf-8-sig')
        df['날짜'] = pd.to_datetime(df['날짜'])
        print(f'✅ 업로드 데이터 로드: {len(df)}행')
    else:
        print('샘플 데이터 사용')
except: pass

print(f'분석 대상: {len(df)}행')
df.head(3)


## Step 3: OEE 계산

In [ ]:
# OEE 3요소 계산
df['가동률'] = df['가동시간'] / df['계획시간'] * 100
이상_시간 = df.get('이상시간', 0)
df['실가동'] = df['가동시간'] - 이상_시간
# 성능률: 실가동 중 표준 대비 실제 생산 (간이 계산)
df['성능률'] = (df['총생산량'] / (df['실가동'] * 2)).clip(0, 1) * 100
df['양품률'] = (df['총생산량'] - df['불량수']) / df['총생산량'] * 100
df['OEE'] = df['가동률'] * df['성능률'] * df['양품률'] / 10000

avg_oee = df['OEE'].mean()
target = FACTORY['oee_target']
print(f'📊 평균 OEE: {avg_oee:.1f}% (목표 {target}%)')
print(f'가동률: {df["가동률"].mean():.1f}% | 성능률: {df["성능률"].mean():.1f}% | 양품률: {df["양품률"].mean():.1f}%')
print(f'목표 달성: {"✅" if avg_oee >= target else "🔄 개선 필요"}')


## Step 4: 시각화

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
df.set_index('날짜')['OEE'].plot(ax=axes[0,0], title='OEE 일별 추이', color='steelblue')
axes[0,0].axhline(target, color='orange', linestyle='--', label=f'목표({target}%)')
axes[0,0].legend()

metrics = ['가동률', '성능률', '양품률']
means = [df[m].mean() for m in metrics]
axes[0,1].bar(metrics, means, color=['#3498db','#2ecc71','#e74c3c'])
axes[0,1].axhline(target, color='orange', linestyle='--')
axes[0,1].set_title('OEE 3요소 평균')
axes[0,1].set_ylabel('%')

df.set_index('날짜')['불량수'].plot(ax=axes[1,0], title='불량수 추이', color='salmon')
df.set_index('날짜')[['가동률','양품률']].plot(ax=axes[1,1], title='가동률 vs 양품률')

plt.suptitle(f'{FACTORY["name"]} OEE 분석 보고서', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('oee_report.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 차트 저장 완료')


## Step 5: 다운로드

In [ ]:
df.to_csv('oee_results.csv', index=False, encoding='utf-8-sig')
from google.colab import files
files.download('oee_results.csv')
files.download('oee_report.png')
print('✅ 다운로드 완료!')


---
## 🔥 확장 과제
1. 라인별 OEE를 비교하는 다중 라인 버전
2. World Class OEE (85%) 기준선을 추가해서 격차 분석
3. `oee_calculator.py`를 CLI에서 실행해보기